In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__notebook__.ipynb
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/df_filtered.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__output__.json
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/train.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/test.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/custom.css


In [2]:
import os
import glob

for path, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        print(os.path.join(path, f))

/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__notebook__.ipynb
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/df_filtered.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/__output__.json
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/train.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/test.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/custom.css


In [3]:
import pandas as pd

BASE = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/'

train = pd.read_csv(BASE + 'train.csv')
test  = pd.read_csv(BASE + 'test.csv')

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"\nColumns: {train.columns.tolist()}")
train.head(3)

Train : (175802, 6)
Test  : (18083, 6)

Columns: ['UserId', 'ProductId', 'Score', 'Time', 'Summary', 'Text']


,UserId,ProductId,Score,Time,Summary,Text
0,A18ECVX2RJ7HUE,B001GVISJM,4,2010-11-05,fresh and greasy!,good flavor! these came securely packed... the...
1,A2MUGFV2TDQ47K,B001GVISJM,5,2010-03-12,Strawberry Twizzlers - Yummy,The Strawberry Twizzlers are my guilty pleasur...
2,A2A9X58G2GTBLP,B001GVISJM,5,2011-12-23,GREAT SWEET CANDY!,"Twizzlers, Strawberry my childhood favorite ca..."


In [4]:
!pip install scikit-surprise -q

from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate
from surprise import accuracy
import warnings
warnings.filterwarnings('ignore')

print("Surprise imported successfully")

Surprise imported successfully


In [5]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split as surprise_split

# Surprise expects ratings in a specific format
reader = Reader(rating_scale=(1, 5))

# Load only the 3 columns Surprise needs
train_data = Dataset.load_from_df(train[['UserId', 'ProductId', 'Score']], reader)
trainset = train_data.build_full_trainset()

print(f"Users in trainset    : {trainset.n_users:,}")
print(f"Products in trainset : {trainset.n_items:,}")
print(f"Ratings in trainset  : {trainset.n_ratings:,}")

Users in trainset    : 20,371
Products in trainset : 16,223
Ratings in trainset  : 175,802


In [6]:
from surprise import SVD
import time

svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)

start = time.time()
svd.fit(trainset)
elapsed = time.time() - start

print(f"SVD trained in {elapsed:.1f} seconds")
print(f"Latent factors : 50")
print(f"Epochs         : 20")

SVD trained in 1.8 seconds
Latent factors : 50
Epochs         : 20


In [7]:
from surprise import accuracy

# Build testset from our temporal test data
test_data = Dataset.load_from_df(test[['UserId', 'ProductId', 'Score']], reader)
testset = test_data.build_full_trainset().build_testset()

# Generate predictions
predictions = svd.test(testset)

# Compute metrics
rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions, verbose=False)

print(f"SVD Results on Test Set")
print(f"=======================")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"Total predictions : {len(predictions):,}")


SVD Results on Test Set
RMSE : 1.1410
MAE  : 0.8680
Total predictions : 18,083


In [8]:
import numpy as np

global_mean = train['Score'].mean()
user_mean   = train.groupby('UserId')['Score'].mean()
item_mean   = train.groupby('ProductId')['Score'].mean()

# Baseline 1: always predict global mean
test['pred_global'] = global_mean

# Baseline 2: predict user's average rating
test['pred_user'] = test['UserId'].map(user_mean).fillna(global_mean)

# Baseline 3: predict item's average rating
test['pred_item'] = test['ProductId'].map(item_mean).fillna(global_mean)

def rmse(actual, predicted):
    return np.sqrt(((actual - predicted) ** 2).mean())

def mae(actual, predicted):
    return (actual - predicted).abs().mean()

actual = test['Score']

print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8}")
print("-" * 38)
print(f"{'Global Mean':<20} {rmse(actual, test['pred_global']):>8.4f} {mae(actual, test['pred_global']):>8.4f}")
print(f"{'User Mean':<20} {rmse(actual, test['pred_user']):>8.4f} {mae(actual, test['pred_user']):>8.4f}")
print(f"{'Item Mean':<20} {rmse(actual, test['pred_item']):>8.4f} {mae(actual, test['pred_item']):>8.4f}")
print(f"{'SVD':<20} {'1.1410':>8} {'0.8680':>8}")

Model                    RMSE      MAE
--------------------------------------
Global Mean            1.2013   0.9428
User Mean              1.3065   0.8989
Item Mean              1.3245   0.9589
SVD                    1.1410   0.8680


In [9]:
from surprise.model_selection import GridSearchCV

param_grid = {
    'n_factors': [25, 50, 100],
    'n_epochs' : [20, 30],
    'lr_all'   : [0.005, 0.010],
    'reg_all'  : [0.02, 0.05]
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs.fit(train_data)

print(f"Best RMSE : {gs.best_score['rmse']:.4f}")
print(f"Best MAE  : {gs.best_score['mae']:.4f}")
print(f"\nBest params (RMSE): {gs.best_params['rmse']}")

Best RMSE : 0.7521
Best MAE  : 0.4502

Best params (RMSE): {'n_factors': 25, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.02}


In [10]:
best_svd = SVD(n_factors=50, n_epochs=30, lr_all=0.01, reg_all=0.02, random_state=42)
best_svd.fit(trainset)

predictions_tuned = best_svd.test(testset)

rmse_tuned = accuracy.rmse(predictions_tuned, verbose=False)
mae_tuned  = accuracy.mae(predictions_tuned, verbose=False)

print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8}")
print("-" * 38)
print(f"{'Global Mean':<20} {'1.2013':>8} {'0.9428':>8}")
print(f"{'SVD (default)':<20} {'1.1410':>8} {'0.8680':>8}")
print(f"{'SVD (tuned)':<20} {rmse_tuned:>8.4f} {mae_tuned:>8.4f}")
print(f"\nImprovement over Global Mean baseline:")
print(f"RMSE : {((1.2013 - rmse_tuned) / 1.2013 * 100):.1f}%")
print(f"MAE  : {((0.9428 - mae_tuned)  / 0.9428 * 100):.1f}%")

Model                    RMSE      MAE
--------------------------------------
Global Mean            1.2013   0.9428
SVD (default)          1.1410   0.8680
SVD (tuned)            1.1557   0.8663

Improvement over Global Mean baseline:
RMSE : 3.8%
MAE  : 8.1%


In [11]:
import pickle

with open('svd_model.pkl', 'wb') as f:
    pickle.dump(best_svd, f)

# Also save predictions for later hybrid combination
import pandas as pd

preds_df = pd.DataFrame([
    {'UserId': p.uid, 'ProductId': p.iid, 'actual': p.r_ui, 'svd_pred': p.est}
    for p in predictions
])

preds_df.to_csv('svd_predictions.csv', index=False)

print(f"Saved svd_model.pkl")
print(f"Saved svd_predictions.csv : {len(preds_df):,} rows")
print(preds_df.head())

Saved svd_model.pkl
Saved svd_predictions.csv : 18,083 rows
           UserId   ProductId  actual  svd_pred
0  A1QAJ948PN36II  B003SE19UK     5.0  4.526652
1  A1QAJ948PN36II  B000G201FG     5.0  4.621373
2  A1QAJ948PN36II  B003SE58KW     5.0  4.454921
3  A1QAJ948PN36II  B003SE19AK     5.0  4.533837
4  A2NP8RNW9T5BQF  B000LKZK7C     1.0  3.859488
